# Notebook 01 — Funciones de ventana

Tema 07. Las **funciones de ventana** (*window functions*) calculan un valor sobre un conjunto de filas relacionadas con la fila actual — **sin colapsarlas** como hace `GROUP BY`. Resuelven de forma directa rankings, totales acumulados, promedios móviles y comparativas entre filas, que de otro modo exigirían *self-joins* o subconsultas correlacionadas.

Cubres: la cláusula `OVER`, `PARTITION BY`, las funciones de ranking `ROW_NUMBER`/`RANK`/`DENSE_RANK`, agregados móviles (sumas acumulativas y promedios móviles) y `LEAD`.

**Contenido de este notebook:**

- [Setup](#setup)
- [La cláusula `OVER`](#la-cláusula-over)
- [`PARTITION BY` — ventanas por grupo](#partition-by--ventanas-por-grupo)
- [Ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`](#ranking-row_number-rank-dense_rank)
- [Top-N por grupo](#top-n-por-grupo)
- [Agregados móviles: acumulado y promedio móvil](#agregados-móviles-acumulado-y-promedio-móvil)
- [`LEAD` — comparar con la fila siguiente](#lead--comparar-con-la-fila-siguiente)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## La cláusula `OVER`

Una función de ventana se escribe `funcion(...) OVER (ventana)`. La diferencia con `GROUP BY`: `GROUP BY` reduce muchas filas a una; una función de ventana **conserva todas las filas** y agrega una columna calculada sobre la "ventana".

Con el `OVER ()` vacío, la ventana es **toda la tabla**:

In [ ]:
%%sql
SELECT
    fs.order_id,
    fs.line_total,
    ROUND(SUM(fs.line_total) OVER (), 2) AS total_general
FROM   northwind_dwh.fact_sales fs
ORDER  BY fs.order_id
LIMIT  10;

Cada fila **conserva** su `line_total` y además ve el `total_general` (la suma de todas las filas). Con `GROUP BY` habrías perdido el detalle fila a fila — esa es la esencia de una función de ventana.

## `PARTITION BY` — ventanas por grupo

`PARTITION BY` divide las filas en grupos y **reinicia la ventana en cada grupo** — como un `GROUP BY` que no colapsa. La función se calcula dentro de cada partición. Aquí: ventas de cada producto, el total de su categoría, y su peso porcentual dentro de la categoría.

In [ ]:
%%sql
WITH ventas_producto AS (
    SELECT dp.category_name, dp.product_name,
           ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_product dp ON dp.product_key = fs.product_key
    GROUP  BY dp.category_name, dp.product_name
)
SELECT
    category_name,
    product_name,
    ventas,
    ROUND(SUM(ventas) OVER (PARTITION BY category_name), 2)               AS ventas_categoria,
    ROUND(100 * ventas / SUM(ventas) OVER (PARTITION BY category_name), 1) AS pct_en_categoria
FROM   ventas_producto
ORDER  BY category_name, ventas DESC
LIMIT  15;

`ventas_categoria` se **repite** en cada fila de la misma categoría: es el total de la partición. Compáralo con `GROUP BY category_name`, que te daría una sola fila por categoría y perderías el detalle por producto.

## Ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`

Las tres numeran filas según el `ORDER BY` de la ventana. Difieren en cómo tratan los **empates**:

| Función | Empates | Secuencia con empate en 2º |
|---|---|---|
| `ROW_NUMBER` | número único siempre | 1, 2, 3, 4 |
| `RANK` | mismo número, **deja huecos** | 1, 2, 2, 4 |
| `DENSE_RANK` | mismo número, **sin huecos** | 1, 2, 2, 3 |

In [ ]:
%%sql
WITH ventas_producto AS (
    SELECT dp.product_name,
           ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_product dp ON dp.product_key = fs.product_key
    GROUP  BY dp.product_name
)
SELECT
    product_name,
    ventas,
    ROW_NUMBER() OVER (ORDER BY ventas DESC) AS row_number,
    RANK()       OVER (ORDER BY ventas DESC) AS rank,
    DENSE_RANK() OVER (ORDER BY ventas DESC) AS dense_rank
FROM   ventas_producto
ORDER  BY ventas DESC
LIMIT  10;

## Top-N por grupo

El patrón más útil del ranking: "los **3 productos más vendidos de cada categoría**". Se combina `PARTITION BY` (un ranking independiente por categoría) con un filtro sobre el número de fila.

Una función de ventana **no se puede filtrar en el `WHERE` de su propia query** (se calcula *después* del `WHERE`/`GROUP BY`), así que se envuelve en una CTE y se filtra por fuera:

In [ ]:
%%sql
WITH ranking AS (
    SELECT
        dp.category_name,
        dp.product_name,
        ROUND(SUM(fs.line_total), 2) AS ventas,
        ROW_NUMBER() OVER (PARTITION BY dp.category_name
                           ORDER BY SUM(fs.line_total) DESC) AS rn
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_product dp ON dp.product_key = fs.product_key
    GROUP  BY dp.category_name, dp.product_name
)
SELECT category_name, product_name, ventas
FROM   ranking
WHERE  rn <= 3
ORDER  BY category_name, ventas DESC;

## Agregados móviles: acumulado y promedio móvil

Al agregar un `ORDER BY` dentro del `OVER`, la ventana se vuelve **acumulativa**: por defecto va desde el inicio hasta la fila actual. El rango exacto se controla con un **frame** `ROWS BETWEEN … AND …`:

- **Suma acumulativa** (*running total*): `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.
- **Promedio móvil de 3** (*rolling average*): `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`.

In [ ]:
%%sql
WITH ventas_mes AS (
    SELECT
        CAST(DATE_TRUNC('month', dd.full_date) AS DATE) AS mes,
        ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
    GROUP  BY DATE_TRUNC('month', dd.full_date)
)
SELECT
    mes,
    ventas,
    ROUND(SUM(ventas) OVER (ORDER BY mes
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS acumulado,
    ROUND(AVG(ventas) OVER (ORDER BY mes
              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)         AS media_movil_3m
FROM   ventas_mes
ORDER  BY mes;

## `LEAD` — comparar con la fila siguiente

`LEAD(col)` trae el valor de la **fila siguiente** según el `ORDER BY` de la ventana — ideal para comparar un período con el que le sigue (análisis de tendencia) sin *self-join*:

In [ ]:
%%sql
WITH ventas_mes AS (
    SELECT
        CAST(DATE_TRUNC('month', dd.full_date) AS DATE) AS mes,
        ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
    GROUP  BY DATE_TRUNC('month', dd.full_date)
)
SELECT
    mes,
    ventas,
    LEAD(ventas) OVER (ORDER BY mes)                     AS ventas_mes_siguiente,
    ROUND(LEAD(ventas) OVER (ORDER BY mes) - ventas, 2)  AS variacion
FROM   ventas_mes
ORDER  BY mes;

La última fila tiene `ventas_mes_siguiente = NULL`: no hay mes posterior. `LEAD` mira hacia adelante; su gemela `LAG` mira hacia atrás y es simétrica.

## Cierre

Lo que cubriste:

| Concepto | Sintaxis |
|---|---|
| Ventana completa | `SUM(x) OVER ()` |
| Ventana por grupo | `SUM(x) OVER (PARTITION BY g)` |
| Ranking | `ROW_NUMBER / RANK / DENSE_RANK OVER (ORDER BY …)` |
| Top-N por grupo | `ROW_NUMBER() OVER (PARTITION BY g ORDER BY …)` + filtro en CTE |
| Acumulado / móvil | `… OVER (ORDER BY … ROWS BETWEEN …)` |
| Fila siguiente | `LEAD(x) OVER (ORDER BY …)` |

---

<p align="center">
<a href="../Tema-06/Readme.md">← Anterior: Tema 06</a> | <a href="Readme.md">Volver al índice</a> | <a href="../Tema-08/Readme.md">Siguiente: Tema 08 →</a>
</p>